[Bron code](https://python.plainenglish.io/hungarian-algorithm-introduction-python-implementation-93e7c0890e15), code schijnt niet goed te zijn kijkend naar de reacties

In [28]:
# @title
import numpy as np

def min_zero_row(zero_mat, mark_zero):

	'''
	The function can be splitted into two steps:
	#1 The function is used to find the row which containing the fewest 0.
	#2 Select the zero number on the row, and then marked the element corresponding row and column as False
	'''

	#Find the row
	min_row = [99999, -1]

	for row_num in range(zero_mat.shape[0]):
		if np.sum(zero_mat[row_num] == True) > 0 and min_row[0] > np.sum(zero_mat[row_num] == True):
			min_row = [np.sum(zero_mat[row_num] == True), row_num]

	# Marked the specific row and column as False
	zero_index = np.where(zero_mat[min_row[1]] == True)[0][0]
	mark_zero.append((min_row[1], zero_index))
	zero_mat[min_row[1], :] = False
	zero_mat[:, zero_index] = False

def mark_matrix(mat):

	'''
	Finding the returning possible solutions for LAP problem.
	'''

	#Transform the matrix to boolean matrix(0 = True, others = False)
	cur_mat = mat
	zero_bool_mat = (cur_mat == 0)
	zero_bool_mat_copy = zero_bool_mat.copy()

	#Recording possible answer positions by marked_zero
	marked_zero = []
	while (True in zero_bool_mat_copy):
		min_zero_row(zero_bool_mat_copy, marked_zero)

	#Recording the row and column positions seperately.
	marked_zero_row = []
	marked_zero_col = []
	for i in range(len(marked_zero)):
		marked_zero_row.append(marked_zero[i][0])
		marked_zero_col.append(marked_zero[i][1])

	#Step 2-2-1
	non_marked_row = list(set(range(cur_mat.shape[0])) - set(marked_zero_row))

	marked_cols = []
	check_switch = True
	while check_switch:
		check_switch = False
		for i in range(len(non_marked_row)):
			row_array = zero_bool_mat[non_marked_row[i], :]
			for j in range(row_array.shape[0]):
				#Step 2-2-2
				if row_array[j] == True and j not in marked_cols:
					#Step 2-2-3
					marked_cols.append(j)
					check_switch = True

		for row_num, col_num in marked_zero:
			#Step 2-2-4
			if row_num not in non_marked_row and col_num in marked_cols:
				#Step 2-2-5
				non_marked_row.append(row_num)
				check_switch = True
	#Step 2-2-6
	marked_rows = list(set(range(mat.shape[0])) - set(non_marked_row))

	return(marked_zero, marked_rows, marked_cols)

def adjust_matrix(mat, cover_rows, cover_cols):
	cur_mat = mat
	non_zero_element = []

	#Step 4-1
	for row in range(len(cur_mat)):
		if row not in cover_rows:
			for i in range(len(cur_mat[row])):
				if i not in cover_cols:
					non_zero_element.append(cur_mat[row][i])
	min_num = min(non_zero_element)

	#Step 4-2
	for row in range(len(cur_mat)):
		if row not in cover_rows:
			for i in range(len(cur_mat[row])):
				if i not in cover_cols:
					cur_mat[row, i] = cur_mat[row, i] - min_num
	#Step 4-3
	for row in range(len(cover_rows)):
		for col in range(len(cover_cols)):
			cur_mat[cover_rows[row], cover_cols[col]] = cur_mat[cover_rows[row], cover_cols[col]] + min_num
	return cur_mat

def hungarian_algorithm(mat):
	dim = mat.shape[0]
	cur_mat = mat

	#Step 1 - Every column and every row subtract its internal minimum
	for row_num in range(mat.shape[0]):
		cur_mat[row_num] = cur_mat[row_num] - np.min(cur_mat[row_num])

	for col_num in range(mat.shape[1]):
		cur_mat[:,col_num] = cur_mat[:,col_num] - np.min(cur_mat[:,col_num])
	zero_count = 0
	while zero_count < dim:
		#Step 2 & 3
		ans_pos, marked_rows, marked_cols = mark_matrix(cur_mat)
		zero_count = len(marked_rows) + len(marked_cols)

		if zero_count < dim:
			cur_mat = adjust_matrix(cur_mat, marked_rows, marked_cols)

	return ans_pos

def ans_calculation(mat, pos):
	total = 0
	ans_mat = np.zeros((mat.shape[0], mat.shape[1]))
	for i in range(len(pos)):
		total += mat[pos[i][0], pos[i][1]]
		ans_mat[pos[i][0], pos[i][1]] = mat[pos[i][0], pos[i][1]]
	return total, ans_mat

def main():

	'''Hungarian Algorithm:
	Finding the minimum value in linear assignment problem.
	Therefore, we can find the minimum value set in net matrix
	by using Hungarian Algorithm. In other words, the maximum value
	and elements set in cost matrix are available.'''

	#The matrix who you want to find the minimum sum
	cost_matrix = np.array([[7, 6, 2, 9, 2],
				[6, 2, 1, 3, 9],
				[5, 6, 8, 9, 5],
				[6, 8, 5, 8, 6],
				[9, 5, 6, 4, 7]])
	ans_pos = hungarian_algorithm(cost_matrix.copy())#Get the element position.
	ans, ans_mat = ans_calculation(cost_matrix, ans_pos)#Get the minimum or maximum value and corresponding matrix.

	#Show the result
	print(f"Linear Assignment problem result: {ans:.0f}\n{ans_mat}")

	#If you want to find the maximum value, using the code as follows:
	#Using maximum value in the cost_matrix and cost_matrix to get net_matrix
	profit_matrix = np.array([[7, 6, 2, 9, 2],
				[6, 2, 1, 3, 9],
				[5, 6, 8, 9, 5],
				[6, 8, 5, 8, 6],
				[9, 5, 6, 4, 7]])
	max_value = np.max(profit_matrix)
	cost_matrix = max_value - profit_matrix
	ans_pos = hungarian_algorithm(cost_matrix.copy())#Get the element position.
	ans, ans_mat = ans_calculation(profit_matrix, ans_pos)#Get the minimum or maximum value and corresponding matrix.
	#Show the result
	print(f"Linear Assignment problem result: {ans:.0f}\n{ans_mat}")

if __name__ == '__main__':
	main()

Linear Assignment problem result: 18
[[0. 0. 0. 0. 2.]
 [0. 2. 0. 0. 0.]
 [5. 0. 0. 0. 0.]
 [0. 0. 5. 0. 0.]
 [0. 0. 0. 4. 0.]]
Linear Assignment problem result: 43
[[0. 0. 0. 9. 0.]
 [0. 0. 0. 0. 9.]
 [0. 0. 8. 0. 0.]
 [0. 8. 0. 0. 0.]
 [9. 0. 0. 0. 0.]]


[Bron code](https://www.geeksforgeeks.org/dsa/hungarian-algorithm-assignment-problem-set-1-introduction/) , volgensmij is dit platform best wel betrouwbaar, wordt ook gebruikt in sommige thesis als bron.

In [34]:
# @title
from collections import deque
import sys

def labelIt(cost, lx):
    n = len(cost)
    for i in range(n):
        for j in range(n):
            lx[i] = max(lx[i], cost[i][j])

def addTree(x, prevX, inTreeX, prev, slack, slackX, lx, ly, cost):
    inTreeX[x] = True
    prev[x] = prevX
    for y in range(len(slack)):
        if lx[x] + ly[y] - cost[x][y] < slack[y]:
            slack[y] = lx[x] + ly[y] - cost[x][y]
            slackX[y] = x

def updateLabels(inTreeX, inTreeY, slack, lx, ly):
    n = len(slack)

    delta = sys.maxsize

    for y in range(n):
        if not inTreeY[y]:
            delta = min(delta, slack[y])

    for x in range(n):
        if inTreeX[x]:
            lx[x] -= delta

    for y in range(n):
        if inTreeY[y]:
            ly[y] += delta

    for y in range(n):
        if not inTreeY[y]:
            slack[y] -= delta

def augment(cost, match, inTreeX, inTreeY, prev, xy, yx, slack, slackX, lx, ly):

    # augmenting path algorithm
    n = len(cost)

    # check if we have found a perfect matching
    if match[0] == n:
        return

    x = y = root = 0
    q = deque()

    # find root of tree
    for i in range(n):
        if xy[i] == -1:
            root = i
            q.append(root)
            prev[i] = -2
            inTreeX[i] = True
            break

    # initialize slack
    for i in range(n):
        slack[i] = lx[root] + ly[i] - cost[root][i]
        slackX[i] = root

    # BFS to find augmenting path
    while True:

        # building tree with BFS cycle
        while q:
            x = q.popleft()

            #iterate through all edges in equality graph
            for y in range(n):
                if lx[x] + ly[y] - cost[x][y] == 0 and not inTreeY[y]:

                    # if y is an exposed vertex in Y
                    # found, so augmenting path exists
                    if yx[y] == -1:
                        x = slackX[y]
                        break
                    else:
                        # else just add y to inTreeY
                        inTreeY[y] = True

                        # add vertex yx[y], which is
                        # matched with y, to the queue
                        q.append(yx[y])

                        # add edges (x, y) and (y, yx[y]) to the tree
                        addTree(yx[y], x, inTreeX, prev, slack, slackX, lx, ly, cost)
            if y < n:
                break

        # augmenting path found
        if y < n:
            break

        # else improve labeling
        updateLabels(inTreeX, inTreeY, slack, lx, ly)

        for y in range(n):
            if not inTreeY[y] and slack[y] == 0:
                if yx[y] == -1:
                    x = slackX[y]
                    break
                else:
                    inTreeY[y] = True
                    if not inTreeX[yx[y]]:
                        q.append(yx[y])
                        addTree(yx[y], slackX[y], inTreeX, prev, slack, slackX, lx, ly, cost)
        if y < n:
            break

    if y < n:
        # augmenting path found
        match[0] += 1

        # update xy and yx
        cx = x
        cy = y
        while cx != -2:
            ty = xy[cx]
            xy[cx] = cy
            yx[cy] = cx
            cx = prev[cx]
            cy = ty

        # reset inTreeX and inTreeY
        for i in range(n):
            inTreeX[i] = False
            inTreeY[i] = False

        # recall function, go to step 1 of the algorithm
        augment(cost, match, inTreeX, inTreeY, prev, xy, yx, slack, slackX, lx, ly)

def findMinCost(cost):
    n = len(cost)

    # convert cost matrix to profit matrix
    # by multiplying each element by -1
    for i in range(n):
        for j in range(n):
            cost[i][j] = -1 * cost[i][j]

    # to store the results
    result = 0

    # number of vertices in current matching
    match = [0]

    xy = [-1] * n
    yx = [-1] * n
    lx = [0] * n
    ly = [0] * n
    slack = [0] * n
    slackX = [0] * n
    prev = [0] * n

    inTreeX = [False] * n
    inTreeY = [False] * n

    labelIt(cost, lx)

    augment(cost, match, inTreeX, inTreeY, prev, xy, yx, slack, slackX, lx, ly)

    for i in range(n):
        result += cost[i][xy[i]]
    return -1 * result

if __name__ == "__main__":
    cost = [
        [2500, 4000, 3500],
        [6000, 8000, 15500],
        [2000, 4000, 2500]
    ]
    print(findMinCost(cost))

21500
